In this lab, we'll practice basic emootion recognition models using deep learning and discuss proper ways of model validation. This material was prepared by Minhyung Kim, KAIST.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Homework

# 데이터셋

In [4]:
import os

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE_PATH   = '/content/drive/MyDrive/Data Science TA (Course Work)/Dataset/'
LABEL_PATH  = BASE_PATH + 'self_annotations/'
SIGNAL_PATH = BASE_PATH + 'e4_data/' # e4_data sensing_data

def load_e4_csv(filepath):
    """Load custom E4 CSV format: header row, columns include timestamp (ms) and value"""
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()

    # timestamps are in milliseconds → compute sample rate from intervals
    timestamps = pd.to_numeric(df['timestamp'], errors='coerce')
    values     = pd.to_numeric(df['value'],     errors='coerce')

    # drop any NaN rows
    mask = timestamps.notna() & values.notna()
    timestamps = timestamps[mask].values
    values     = values[mask].values

    # sample rate = 1000ms / avg interval between samples
    avg_interval_ms = np.diff(timestamps).mean()
    sample_rate = round(1000.0 / avg_interval_ms)

    return values, sample_rate

def segment_signal(data, sample_rate, window_sec=5):
    """Split continuous signal into non-overlapping 5-second windows"""
    window_size = int(sample_rate * window_sec)
    if window_size == 0 or len(data) < window_size:
        return np.array([])
    n_windows = len(data) // window_size
    return np.array([data[i*window_size:(i+1)*window_size] for i in range(n_windows)])

# ── participants with no signal folder ──────────────────────────────
MISSING_SIGNAL = set()

dataset = {}
for fname in sorted(os.listdir(LABEL_PATH)):
    if not fname.endswith('.self.csv'):
        continue

    pid    = fname.replace('.self.csv', '')
    num_id = pid.replace('P', '').replace('p', '')
    sig_dir = os.path.join(SIGNAL_PATH, num_id)

    df_label = pd.read_csv(os.path.join(LABEL_PATH, fname))

    signals = {}
    if os.path.exists(sig_dir):
        try:
            bvp,  bvp_sr  = load_e4_csv(os.path.join(sig_dir, 'E4_BVP.csv'))
            eda,  eda_sr  = load_e4_csv(os.path.join(sig_dir, 'E4_EDA.csv'))
            temp, temp_sr = load_e4_csv(os.path.join(sig_dir, 'E4_TEMP.csv'))
            hr,   hr_sr   = load_e4_csv(os.path.join(sig_dir, 'E4_HR.csv'))

            signals = {
                'bvp':  segment_signal(bvp,  bvp_sr),
                'ecg':  segment_signal(hr,   hr_sr),   # HR used as ECG proxy
                'eda':  segment_signal(eda,  eda_sr),
                'temp': segment_signal(temp, temp_sr),
            }
        except Exception as e:
            print(f"❌ Error loading signals for {pid}: {e}")
    else:
        MISSING_SIGNAL.add(pid)

    dataset[pid] = {
        'labels': {
            'self_arousal': list(df_label['arousal']),
            'self_valence': list(df_label['valence']),
        },
        'data': signals
    }

print(f"⚠️  No signal folder for: {sorted(MISSING_SIGNAL)}")
print(f"✅  Loaded {len(dataset) - len(MISSING_SIGNAL)} participants with signals\n")

# ── sanity check ────────────────────────────────────────────────────
for sig, arr in dataset['P1']['data'].items():
    print(f"  P1 {sig}: {arr.shape}  (windows × samples_per_window)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️  No signal folder for: ['P2', 'P3', 'P6', 'P7']
✅  Loaded 28 participants with signals

  P1 bvp: (314, 320)  (windows × samples_per_window)
  P1 ecg: (313, 5)  (windows × samples_per_window)
  P1 eda: (314, 20)  (windows × samples_per_window)
  P1 temp: (314, 20)  (windows × samples_per_window)


## Q1 Build a multi-modal model (1.5 pt)

We learned how to sample signals and make model in the last session. Do the following tasks.

(1) Downsample ```Participant8 (P8)```'s BVP 64Hz) signal to ```4Hz``` and merge it with temperature data (```4Hz```). This can be considered as "early fusion."

(2) Reshape to (20, 2) This means that there are two channels where each channel has 20 samples (5 seconds * 4 Hz = 20 samples). It considers 20 samples in a single LSTM layer (and each input is two dimensional).

(3) Create a two-layer LSTM model where each LSTM layer has 64 hidden units

(4) Train your model

(5) Save the model to your path

(6) Reason why you have specific numbers of model parameters at each layer


In [200]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras import layers

# https://en.wikipedia.org/wiki/Precision_and_recall
# https://www.kaggle.com/code/kobiljon/precision-and-recall-relationship-f1-score

# metrics
from tensorflow.keras import backend as K

def recall_m(y_true, y_pred):
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        recall = true_positives / (possible_positives + K.epsilon())
        return recall

def precision_m(y_true, y_pred):
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        precision = true_positives / (predicted_positives + K.epsilon())
        return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))


In [198]:
# import pandas as pd
# import numpy as np

# # ── Pull raw signal arrays for P8 ───────────────────────────────
# # dataset['P8']['dattten()   # 64Hz
# bvp_raw  = dataset['P8']['data']['bvp'].flatten()   # 64Hz
# temp_raw = dataset['P8']['data']['temp'].flatten()  # 4Hz
# Y = dataset['P8']['labels']['self_arousal'] # (n_trials, 2)

# print('BVP  samples:', len(bvp_raw),  '→ duration:', len(bvp_raw)/64,  'secs')
# print('Temp samples:', len(temp_raw), '→ duration:', len(temp_raw)/4,  'secs')

# # ── Downsample BVP from 64Hz → 4Hz (rolling mean, window=16) ────
# bvp_df = pd.DataFrame(bvp_raw, columns=['bvp'])
# bvp_df = bvp_df.rolling(16).mean()            # (1) moving average over 16 samples
# bvp_4hz = bvp_df.iloc[list(range(15, len(bvp_df), 16)), :]  # (2) pick every 16th sample
# bvp_4hz = bvp_4hz.reset_index(drop=True)

# # ── Align lengths (in case of rounding mismatch) ─────────────────
# min_len = min(len(bvp_4hz), len(temp_raw))
# bvp_4hz  = bvp_4hz.iloc[:min_len]
# temp_arr = temp_raw[:min_len]
# Y = Y[:min_len]

# print(min_len,len(bvp_4hz),len(temp_arr),len(Y))

# # ── Merge (early fusion) ─────────────────────────────────────────
# df = pd.DataFrame({
#     'bvp':  bvp_4hz['bvp'].values,
#     'temp': temp_arr,
#     'Y' : Y
# })

# print(f'\nDownsampled BVP: {len(bvp_4hz)} samples at 4Hz')
# print(f'Merged shape: {df.shape}  (samples × channels)')
# print(df.head())

BVP  samples: 81600 → duration: 1275.0 secs
Temp samples: 5100 → duration: 1275.0 secs
5100 5100 5100 120


In [201]:
# P8 data
import pandas as pd
import numpy as np
from tensorflow.keras.utils import to_categorical
import pandas as pd
import numpy as np
bvp = dataset['P8']['data']['bvp']          # (n_trials, 320)
temp = dataset['P8']['data']['temp']        # (n_trials, 20)
Y = dataset['P8']['labels']['self_arousal'] # (n_trials, 2)

# Signal과 label의 trial 수 맞추기
n_trials = min(len(bvp), len(temp), len(Y))


bvp = np.array(bvp[:n_trials])
temp = np.array(temp[:n_trials])
Y = np.array(Y[:n_trials])

bvp = bvp[:n_trials]
temp = temp[:n_trials]
Y = Y[:n_trials]

# BVP: 64 Hz → 4 Hz
# 한 trial의 320개 sample을 16개씩 평균내면 20개 sample이 됨
bvp_4hz = bvp.reshape(n_trials, 20, 16).mean(axis=2)

print("Downsampled BVP shape:", bvp_4hz.shape)
print("Temperature shape:", temp.shape)

# # Early fusion: BVP와 Temperature를 feature/channel 축으로 합치기
X = np.stack([bvp_4hz, temp], axis=-1)
Y = np.array(Y)

# 이진으로 만들기
# it just have 3 & 4
Y_binary = np.where(Y == 3,0,1).astype(int)

Y_onehot = to_categorical(Y_binary, num_classes=2)

print("Final X shape:", X.shape)
print("Y shape:", Y_onehot.shape)

Downsampled BVP shape: (120, 20)
Temperature shape: (120, 20)
Final X shape: (120, 20, 2)
Y shape: (120, 2)


In [202]:
Y_binary,Y_onehot

(array([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0]),
 array([[0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1.],
        [0., 1

In [203]:
model = Sequential()
model.add(LSTM(64, return_sequences = True, dropout = 0.2,  input_shape=(20, 2)))
model.add(BatchNormalization()) # https://www.youtube.com/watch?v=sdXfAY_VD58

model.add(LSTM(64, dropout = 0.2))
model.add(BatchNormalization())

model.add(Dense(2, activation='softmax'))

model.summary()

Model: "sequential_29"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_36 (LSTM)                  │ (None, 20, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 20, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_37 (LSTM)                  │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_37          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,818 (198.51 KB)

 Trainable params: 50,562 (197.51 KB)

 Non-trainable params: 256 (1.00 KB)

In [204]:
import os
# save your model
path = '/content/drive/MyDrive/Data Science TA (Course Work)/models/' # your path 'path'
if not os.path.exists(path):
  os.mkdir(path)
else:
  pass

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y_onehot, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.25, random_state = 1)

X_train, X_val, y_train, y_val = X_train.astype(float), X_val.astype(float), y_train.astype(float), y_val.astype(float)

# binary so I changed categorical_crossentropy to binary_crossentropy
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy',f1_m]) # recall_m
hist = model.fit(X_train, y_train, batch_size = 8, epochs = 100, verbose = 1, validation_data = (X_val, y_val))

model.save(path+'model.h5')

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.4722 - f1_m: 0.4722 - loss: 0.8277 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6612
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6111 - f1_m: 0.6111 - loss: 0.7280 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6333
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5694 - f1_m: 0.5694 - loss: 0.6997 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6250
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5694 - f1_m: 0.5694 - loss: 0.7247 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6227
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5833 - f1_m: 0.5833 - loss: 0.7275 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6196
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6250 - f1_m: 0.6250 - loss: 0.6340 - val_accuracy: 0.7083 - val_f1_m: 0.7083 - val_loss: 0.6179
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [205]:
_,acc,f1 = model.evaluate(X_test, y_test)
print("acc: {}%, f1:{}%.".format(round(acc*100,2),round(f1*100,2)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - accuracy: 0.7083 - f1_m: 0.7083 - loss: 0.8439
acc: 70.83%, f1:70.83%.


## [Advanced, Extra Points] Q2 Apply transfer learning (1.5 pt)
Transfer learning is a machine learning method where a model developed for one task is reused as the starting point for training another model for a different task.

It's also known as pre-training and fine-tuning in that pre-trained models are used as a starting point for fine-tuning it using a small number of datasets. This technique is quite popular among computer vision and natural language processing tasks.

In the following, we examine an example of transfer learning. We just provided a code segment to better guide you to use transfer learning. You've already read the reading assignment, and for the detailed implementation issues, please refer to [the Keras web page](https://keras.io/guides/transfer_learning/).

In [207]:
# load pretrained model
model_A = keras.models.load_model(path+'model.h5')

# load weight except the last layer (classifer)
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])

# Add last layer
model_B_on_A.add(keras.layers.Dense(2, activation="sigmoid"))

# Freeze all layers exept the last layer
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

# confile and training
model_B_on_A.compile(loss="binary_crossentropy",
                     optimizer=keras.optimizers.SGD(learning_rate=0.001), #lr=1e-3
                     metrics=["accuracy"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y_onehot, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.25, random_state = 1)

X_train_B, X_valid_B, y_train_B, y_valid_B = X_train.astype(float), X_val.astype(float), y_train.astype(float), y_val.astype(float)

history = model_B_on_A.fit(X_train_B, y_train_B, epochs=4,
                           validation_data=(X_valid_B, y_valid_B))


acc,f1 = model_B_on_A.evaluate(X_test, y_test)
print("acc: {}%, f1:{}%.".format(round(acc*100,2),round(f1*100,2)))


Epoch 1/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 457ms/step - accuracy: 0.3611 - loss: 1.0745 - val_accuracy: 0.3333 - val_loss: 0.8734
Epoch 2/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4583 - loss: 0.9761 - val_accuracy: 0.3333 - val_loss: 0.8704
Epoch 3/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.4167 - loss: 1.0305 - val_accuracy: 0.3750 - val_loss: 0.8665
Epoch 4/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3750 - loss: 0.9516 - val_accuracy: 0.3750 - val_loss: 0.8623
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.2500 - loss: 1.0188
acc: 101.88%, f1:25.0%.


do the following tasks:

(1) Load the P8's model designed in Q1.

(2) Reuse P8's LSTM weights to retrain a classifier only using P9's dataset. We call this model as  ```model 1```.

(3) Train the whole model (LSTM+Classifier) using P9's dataset. We call this model as ```model 2```

(5) Compare the performance of ```model 1``` and ```model 2``` and explain your answer.

(6) (Extra Point: 1 pt) Another useful way of transfer learning is personalization of a global model. For example, in our example, you can leave P8's data for training (just like a LOSO evaluation). And then you can use that model to perform transfer learning using P8's data. If you do this, we'll give you an extra 1 point.


In [216]:
from sklearn.preprocessing import OneHotEncoder

# ohe = OneHotEncoder(sparse=False)

# P8 data
bvp_P9 = dataset['P9']['data']['bvp']          # (n_trials, 320)
temp_P9 = dataset['P9']['data']['temp']        # (n_trials, 20)
Y_P9 = dataset['P9']['labels']['self_arousal'] # (n_trials, 2)

# Signal과 label의 trial 수 맞추기
n_trials_P9 = min(len(bvp_P9), len(temp_P9), len(Y_P9))

bvp_P9 = bvp_P9[:n_trials_P9]
temp_P9 = temp_P9[:n_trials_P9]
Y_P9 = Y_P9[:n_trials_P9]

# BVP: 64 Hz → 4 Hz
# 한 trial의 320개 sample을 16개씩 평균내면 20개 sample이 됨
bvp_4hz_P9 = bvp_P9.reshape(n_trials_P9, 20, 16).mean(axis=2)

print("Downsampled BVP shape:", bvp_4hz_P9.shape)
print("Temperature shape:", temp_P9.shape)

# Early fusion: BVP와 Temperature를 feature/channel 축으로 합치기
X_P9 = np.stack([bvp_4hz_P9, temp_P9], axis=-1)

def one_hot_encoding(Y):
  uniqe_Y = np.unique(Y)
  len_Y = len(Y)
  all_un_len = []

  for _ in range(len_Y):

    len_un = len(uniqe_Y)
    all_un_len.append(len_un)
    maxs = np.max(all_un_len)


  sit = Y - np.min(uniqe_Y)
  Y_encode = np.zeros(maxs)

  Y_onehot = []

  for s in sit:
    Y_encode[s] = 1
    Y_onehot.append(Y_encode)
    Y_encode = np.zeros(maxs)

  fin_Y = np.array(Y_onehot) #.reshape(-1,)

  print("Y shape:", fin_Y.shape)
  print("Y :", fin_Y)
  return fin_Y

Y_onehot = one_hot_encoding(Y_P9)
fin_Y9 = np.array(Y_onehot) #.reshape(-1,)

print("Final X shape:", X.shape)


# # Y_P9 = np.array(Y_P9).reshape(-1, 1)
# # Y_one_P9 = ohe.fit_transform(Y_P9)
# Y_P9_final = np.array(Y_one_P9)
# print("Final X shape:", X_P9.shape)
# print("Y shape:", Y_P9.shape)

# from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_P9, fin_Y9, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.25, random_state = 1)

X_train_P9, X_valid_P9, y_train_P9, y_valid_P9 = X_train.astype(float), X_val.astype(float), y_train.astype(float), y_val.astype(float)

Downsampled BVP shape: (122, 20)
Temperature shape: (122, 20)
Y shape: (122, 4)
Y : [[0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 

In [225]:
# Load model
# (2) Reuse P8's LSTM weights to retrain a classifier only using P9's dataset. We call this model as model 1.
model1 = keras.models.load_model(path+'model.h5')

# Add last layer
# 마지막 classifier 제거
model1 = keras.Sequential(model1.layers[:-1])
model1.add(keras.layers.Dense(4, activation="softmax"))

for layer in model1.layers[:-1]:
    layer.trainable = False

model1.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy',f1_m])

history = model1.fit(X_train_P9, y_train_P9, epochs=100,
                           validation_data=(X_valid_P9, y_valid_P9))

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 289ms/step - accuracy: 0.1944 - f1_m: 0.1611 - loss: 2.3603 - val_accuracy: 0.2400 - val_f1_m: 0.2632 - val_loss: 1.6569
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.2083 - f1_m: 0.2262 - loss: 2.2709 - val_accuracy: 0.2400 - val_f1_m: 0.2500 - val_loss: 1.6207
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.1667 - f1_m: 0.1477 - loss: 2.1307 - val_accuracy: 0.2400 - val_f1_m: 0.2500 - val_loss: 1.5893
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.2500 - f1_m: 0.1749 - loss: 2.0064 - val_accuracy: 0.2800 - val_f1_m: 0.2500 - val_loss: 1.5667
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.2222 - f1_m: 0.1130 - loss: 2.1696 - val_accuracy: 0.3200 - val_f1_m: 0.2424 - val_loss: 1.5473
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.2500 - f1_m: 0.1399 - loss: 2.1017 - val_accuracy: 0.4400 - val_f1_m: 0.2424 - val_loss: 1.5295
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 

In [218]:
y_test

array([[0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]])

In [227]:
_,acc,f1 = model1.evaluate(X_test, y_test)
print("acc: {}%, f1:{}%.".format(round(acc*100,2),round(f1*100,2)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.5200 - f1_m: 0.4324 - loss: 0.9697
acc: 52.0%, f1:43.24%.


In [230]:
# make model 2
# (3) Train the whole model (LSTM+Classifier) using P9's dataset. We call this model as model 2
model2 = keras.models.load_model(path+'model.h5')

# Add last layer
# 마지막 classifier 제거
model2 = keras.Sequential(model2.layers[:-1])

# 새 4-class classifier 추가
model2.add(keras.layers.Dense(4, activation="softmax"))

# if we need want to train all layer of model, we have to un-freeze all layer
for layer in model2.layers[:]:
    layer.trainable = True

model2.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy',f1_m])

#your answer in a textbox >> this mean model2
history = model2.fit(X_train_P9, y_train_P9, epochs=100,
                           validation_data=(X_valid_P9, y_valid_P9))

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 272ms/step - accuracy: 0.1667 - f1_m: 0.0593 - loss: 1.9591 - val_accuracy: 0.4000 - val_f1_m: 0.3404 - val_loss: 1.9701
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2639 - f1_m: 0.1779 - loss: 1.6184 - val_accuracy: 0.4400 - val_f1_m: 0.3111 - val_loss: 1.9612
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3472 - f1_m: 0.2773 - loss: 1.3066 - val_accuracy: 0.3600 - val_f1_m: 0.3043 - val_loss: 1.9363
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3333 - f1_m: 0.1271 - loss: 1.3576 - val_accuracy: 0.3200 - val_f1_m: 0.2927 - val_loss: 1.9110
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4306 - f1_m: 0.2734 - loss: 1.2916 - val_accuracy: 0.3200 - val_f1_m: 0.2927 - val_loss: 1.9132
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3889 - f1_m: 0.2437 - loss: 1.3673 - val_accuracy: 0.3200 - val_f1_m: 0.2791 - val_loss: 1.8809
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s

In [231]:
_,acc,f1 = model2.evaluate(X_test, y_test)
print("acc: {}%, f1:{}%.".format(round(acc*100,2),round(f1*100,2)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.6400 - f1_m: 0.6250 - loss: 0.8382
acc: 64.0%, f1:62.5%.


If I do it well on HW, the performance different because model1 is just fit in P8data and also P9 data is 4 class so when we want to see nice performance on P9, we have to train again. train again or not is make performance different.


(6) (Extra Point: 1 pt) Another useful way of transfer learning is personalization of a global model. For example, in our example, you can leave P8's data for training (just like a LOSO evaluation). And then you can use that model to perform transfer learning using P8's data. If you do this, we'll give you an extra 1 point.

In [232]:
print(len(dataset['P8']['data']['bvp']) / 3)
print("Let's say they are 3 group")

85.0
Let's say they are 3 group


In [233]:
# ── Pull raw signal arrays for P8 ───────────────────────────────
# dataset['P8']['dattten()   # 64Hz
bvp_raw  = dataset['P8']['data']['bvp'].flatten()   # 64Hz
temp_raw = dataset['P8']['data']['temp'].flatten()  # 4Hz

print('BVP  samples:', len(bvp_raw),  '→ duration:', len(bvp_raw)/64,  'secs')
print('Temp samples:', len(temp_raw), '→ duration:', len(temp_raw)/4,  'secs')

# ── Downsample BVP from 64Hz → 4Hz (rolling mean, window=16) ────
bvp_df = pd.DataFrame(bvp_raw, columns=['bvp'])
bvp_df = bvp_df.rolling(16).mean()            # (1) moving average over 16 samples
bvp_4hz = bvp_df.iloc[list(range(15, len(bvp_df), 16)), :]  # (2) pick every 16th sample
bvp_4hz = bvp_4hz.reset_index(drop=True)

# ── Align lengths (in case of rounding mismatch) ─────────────────
min_len = min(len(bvp_4hz), len(temp_raw))
bvp_4hz  = bvp_4hz.iloc[:min_len]
temp_arr = temp_raw[:min_len]

# ── Merge (early fusion) ─────────────────────────────────────────
df = pd.DataFrame({
    'bvp':  bvp_4hz['bvp'].values,
    'temp': temp_arr
})

print(f'\nDownsampled BVP: {len(bvp_4hz)} samples at 4Hz')
print(f'Merged shape: {df.shape}  (samples × channels)')
print(df.head())

BVP  samples: 81600 → duration: 1275.0 secs
Temp samples: 5100 → duration: 1275.0 secs

Downsampled BVP: 5100 samples at 4Hz
Merged shape: (5100, 2)  (samples × channels)
          bvp   temp
0   -0.003750  30.99
1    2.164375  30.99
2   11.654375  30.99
3   49.225000  30.99
4  160.436250  30.99


In [234]:
df

,bvp,temp
0,-0.003750,30.99
1,2.164375,30.99
2,11.654375,30.99
3,49.225000,30.99
4,160.436250,30.99
...,...,...
5095,4.625625,30.21
5096,8.533750,30.23
5097,16.065000,30.23
5098,15.219375,30.23


In [235]:
import numpy as np
from scipy.signal import resample # Added this import

# !!note : make sure to go through skewness of each participant's dataset before doing LOSO

# please don't use this one, make another one!
import numpy as np
from scipy.signal import resample


def make_LOSO_dataset(currentID, dataset, modal, label):
    LOSO_dataset = {}

    data = dataset[currentID]["data"]
    labels = dataset[currentID]["labels"]

    modal_data = np.asarray(data[modal])
    modal_label = np.asarray(labels[label])

    len_data = min(len(modal_data), len(modal_label))

    modal_data = modal_data[:len_data]
    modal_label = modal_label[:len_data]

    n_group = 3
    valid_pids = {}

    # 1. Split P8 data into 5 groups
    for i in range(n_group):
        start = (len_data * i) // n_group
        end = (len_data * (i + 1)) // n_group

        valid_pids[f"group_{i}"] = {
            "data": modal_data[start:end],
            "label": modal_label[start:end]
        }

    # 2. Leave one group out
    for test_group in valid_pids:
        train_X_list = []
        train_Y_list = []

        test_X_current = None
        test_Y_current = None

        target_samples_per_window = 5 * 17  # 85

        for group_name in valid_pids:
            current_group_data = valid_pids[group_name]["data"]
            current_group_label = valid_pids[group_name]["label"]

            current_group_data = np.asarray(current_group_data)
            current_group_label = np.asarray(current_group_label)

            # If data is 1D, make it 2D
            if current_group_data.ndim == 1:
                current_group_data = current_group_data.reshape(1, -1)

            actual_samples_per_window = current_group_data.shape[1]

            processed_modal_data = current_group_data

            # Resample if needed
            if actual_samples_per_window != target_samples_per_window:
                resampled_windows = []

                for window in current_group_data:
                    resampled_window = resample(window, target_samples_per_window)
                    resampled_windows.append(resampled_window)

                processed_modal_data = np.array(resampled_windows)

                print(
                    f"Resampled {group_name} data for participant {currentID} "
                    f"from {actual_samples_per_window} to {target_samples_per_window} samples per window."
                )

            # Reshape X for LSTM input
            reshaped_X = processed_modal_data.reshape(-1, 5 , 17)
            current_Y = current_group_label

            # Match X and Y length
            if len(reshaped_X) != len(current_Y):
                min_len = min(len(reshaped_X), len(current_Y))
                reshaped_X = reshaped_X[:min_len]
                current_Y = current_Y[:min_len]

                print(
                    f"Warning: {currentID} {group_name} data and {label} labels "
                    f"had inconsistent lengths. Truncating to {min_len} samples."
                )

            if group_name == test_group:
                test_X_current = reshaped_X
                test_Y_current = current_Y
            else:
                train_X_list.append(reshaped_X)
                train_Y_list.append(current_Y)

        # Important: this part should be OUTSIDE the inner loop
        if test_X_current is None or test_Y_current is None:
            print(f"Warning: Test data for {test_group} could not be prepared. Skipping this fold.")
            continue

        if not train_X_list or not train_Y_list:
            print(f"Warning: No training data available for {test_group}. Skipping this fold.")
            continue

        train_X_combined = np.concatenate(train_X_list, axis=0)
        train_Y_combined = np.concatenate(train_Y_list, axis=0)

        LOSO_dataset[test_group] = {
            "train": {
                "X": train_X_combined,
                "Y": train_Y_combined
            },
            "test": {
                "X": test_X_current,
                "Y": test_Y_current
            }
        }

    return LOSO_dataset

P8_LOSO_dataset=make_LOSO_dataset(currentID = "P8",dataset = dataset, modal = 'bvp', label ='self_arousal')

Resampled group_0 data for participant P8 from 320 to 85 samples per window.
Resampled group_1 data for participant P8 from 320 to 85 samples per window.
Resampled group_2 data for participant P8 from 320 to 85 samples per window.
Resampled group_0 data for participant P8 from 320 to 85 samples per window.
Resampled group_1 data for participant P8 from 320 to 85 samples per window.
Resampled group_2 data for participant P8 from 320 to 85 samples per window.
Resampled group_0 data for participant P8 from 320 to 85 samples per window.
Resampled group_1 data for participant P8 from 320 to 85 samples per window.
Resampled group_2 data for participant P8 from 320 to 85 samples per window.


In [236]:
P8_LOSO_dataset['group_0'].keys()

dict_keys(['train', 'test'])

In [237]:
def one_hot_encoding(Y):
  uniqe_Y = np.unique(Y)
  Y_encode = np.zeros(len(uniqe_Y))
  mins = np.min(Y_encode)
  len_Y = len(Y)
  sit = Y - np.min(uniqe_Y)
  Y_onehot = []
  for s in sit:
    Y_encode[s] = 1
    Y_onehot.append(Y_encode)
    Y_encode = np.zeros(len(uniqe_Y))

  fin_Y = np.array(Y_onehot) #.reshape(-1,)

  print("Y shape:", fin_Y.shape)
  print("Y :", fin_Y)
  return fin_Y


for g in P8_LOSO_dataset.keys():
  for tt in P8_LOSO_dataset[g].keys():
    P8_LOSO_dataset[g][tt]["Y"] = one_hot_encoding(P8_LOSO_dataset[g][tt]["Y"])

Y shape: (80, 2)
Y : [[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 0.]]
Y shape: (40, 2)
Y : [[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0

In [238]:
model_P8 = Sequential()
model_P8.add(LSTM(16, return_sequences = True, dropout = 0.2, input_shape = (5,64)))
model_P8.add(BatchNormalization()) # https://www.youtube.com/watch?v=sdXfAY_VD58
model_P8.add(LSTM(16, dropout = 0.2))
model_P8.add(BatchNormalization())
model_P8.add(Dense(2, activation='softmax'))

model_P8.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_40"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_38 (LSTM)                  │ (None, 5, 16)          │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_38          │ (None, 5, 16)          │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_39 (LSTM)                  │ (None, 16)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_39          │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,458 (29.13 KB)

 Trainable params: 7,394 (28.88 KB)

 Non-trainable params: 64 (256.00 B)

In [239]:
# This is a very time consuming block because you are training for all participants.

acc_list, f1_list = [],[]
for group in P8_LOSO_dataset.keys():
    print("test group: " + group)
    x_train,x_test = P8_LOSO_dataset[group]['train']['X'],P8_LOSO_dataset[group]['test']['X']
    y_train,y_test = P8_LOSO_dataset[group]['train']['Y'],P8_LOSO_dataset[group]['test']['Y']

    x_train,x_test = x_train.astype(float),x_test.astype(float)
    y_train,y_test = y_train.astype(float),y_test.astype(float)

    model_P8.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy',f1_m])
    # print(x_train, y_train)
    hist=model_P8.fit(x_train, y_train,epochs = 100 ,validation_data = (x_test, y_test))
    loss_met_result = model_P8.evaluate(x_test,y_test, return_dict=True)
    acc,f1 = loss_met_result["accuracy"] , loss_met_result["f1_m"]
    print("Result of {} : accuracy: {}, f1: {}".format(group,acc*100,f1*100))
    acc_list.append(acc*100)
    f1_list.append(f1*100)
print("Result")
print("acc: %.2f%% (+/- %.2f%%)" % (np.mean(acc_list), np.std(acc_list)))
print("f1: %.2f%% (+/- %.2f%%)" % (np.mean(f1_list), np.std(f1_list)))

test group: group_0
Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 291ms/step - accuracy: 0.4750 - f1_m: 0.4583 - loss: 1.2481 - val_accuracy: 0.5250 - val_f1_m: 0.4687 - val_loss: 0.6924
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5125 - f1_m: 0.4896 - loss: 1.1059 - val_accuracy: 0.5250 - val_f1_m: 0.5156 - val_loss: 0.6848
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4625 - f1_m: 0.4896 - loss: 1.1991 - val_accuracy: 0.6000 - val_f1_m: 0.6094 - val_loss: 0.6796
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5125 - f1_m: 0.5208 - loss: 1.1103 - val_accuracy: 0.6000 - val_f1_m: 0.6562 - val_loss: 0.6757
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5375 - f1_m: 0.5104 - loss: 1.0484 - val_accuracy: 0.6250 - val_f1_m: 0.6719 - val_loss: 0.6727
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5500 - f1_m: 0.5417 - loss: 0.9689 - val_accuracy: 0.6750 - val_f1_m: 0.7031 - val_loss: 0.6697
Epoch 7/100
3/3 ━━━